In [10]:
import torch
import os
import torch.nn as nn
import numpy as np
from torch.nn.utils import clip_grad_norm

# I. Define dictionary

In [11]:
class Dictionary(object):
    def __init__(self):
        self.word2idx, self.idx2word, self.idx = {}, {}, 0

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word],  self.idx2word[self.idx] = self.idx, word
            self.idx += 1

    def __len__(self):
        return len(self.word2idx)


class TextProcess(object):

    def __init__(self):
        self.dictionary = Dictionary()

    def get_data(self, path, batch_size=20):
        # Đếm số tokens và xây dictionary
        with open(path, 'r') as f:
            tokens = 0
            for line in f:
                words = line.split() + ['<eos>']
                tokens += len(words)
                for word in words: self.dictionary.add_word(word)

        # Tạo tensor rỗng kích thước = số tokens
        rep_tensor = torch.zeros(tokens, dtype=torch.long)

        # Gán chỉ số của từng word vào tensor
        index = 0
        with open(path, 'r') as f:
            for line in f:
                words = line.split() + ['<eos>']
                for word in words: rep_tensor[index], index = self.dictionary.word2idx[word], index+1
        print(f"Có tổng {rep_tensor.shape[0]} từ, mà chia ra {batch_size}.")
        # Tính số batch, cắt bỏ phần dư thừa và reshape thành (batch_size, num_batches)
        num_batches = rep_tensor.shape[0] // batch_size
        rep_tensor = rep_tensor[:num_batches * batch_size].view(batch_size, -1)
        return rep_tensor

# II. Parameter

In [12]:
embed_size = 128    #Input features to the LSTM
hidden_size = 1024  #Number of LSTM units
num_layers = 1
num_epochs = 20
batch_size = 20
timesteps = 30
learning_rate = 0.002

In [13]:
corpus = TextProcess()
rep_tensor = corpus.get_data('/content/alice.txt', batch_size) # Đây là dữ liệu đi train
vocab_size = len(corpus.dictionary)
num_batches = rep_tensor.shape[1] // timesteps

Có tổng 29686 từ.


In [14]:
print(rep_tensor.shape)
print(vocab_size)
print(num_batches)

rep_tensor

torch.Size([20, 1484])
5290
49


tensor([[   0,    1,    2,  ...,  203,  571,    5],
        [ 572,    5,    5,  ...,  988,    5,  107],
        [ 117,    3,  609,  ..., 1364, 1010, 1106],
        ...,
        [   3, 3779,    7,  ...,    5,    5, 2412],
        [ 218,   13,    3,  ..., 1286,  112, 5066],
        [ 632,    5,  345,  ...,    3, 5287, 4779]])

# III. Defining and training model

Với RNN/LSTM (chuỗi, NLP, time series) trong Pytorch, đầu vào dùng format (N, L, D) khi khi batch_first=True:


*   N: batch size
*   L: sequence length (timesteps, số từ trong câu hoặc số bước trong chuỗi)
* D: feature size (ở NLP là embed_size)

Ví dụ: batch 32 câu, mỗi câu dài 10 từ, embedding 300 chiều: (32, 10, 300)


In [15]:
class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(TextGenerator, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        x = self.embed(x) # Perform Word Embedding
        out, (h, c) = self.lstm(x, h) # Reshape the input tensor: x = x.view(batch_size,timesteps,embed_size)
        # Reshape the output from (samples,timesteps,output_features) to a shape appropriate for the FC layer
        # (batch_size*timesteps, hidden_size)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))
        # Decode hidden states of all time steps
        out = self.linear(out)
        return out, (h, c)

In [16]:
model = TextGenerator(vocab_size, embed_size, hidden_size, num_layers)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [17]:
def detach(states):
    return [state.detach() for state in states]

In [18]:
accum_steps = 4  # số batch nhỏ để tích lũy gradient trước khi update
effective_batch_size = batch_size // accum_steps  # batch nhỏ hơn

for epoch in range(num_epochs):
    # Reset hidden state ở đầu mỗi epoch
    states = (torch.zeros(num_layers, effective_batch_size, hidden_size),
              torch.zeros(num_layers, effective_batch_size, hidden_size))

    # Shuffle các hàng để mix batch
    perm = torch.randperm(batch_size)
    rep_shuffled = rep_tensor[perm]

    """
    Chia batch hiện tại có size là batch_size thành accum_steps batch nhỏ hơn
    với size là effective_batch_size = batch_size // accum_steps
    để thực hiện gradient accumulation

    Với mỗi hàng trong batch, lại chia thành các window không trùng nhau
    size là timesteps, nghĩa là gồm num_chunks = len(rows) // timesteps
    Mỗi vòng lặp, X là window_ith và y là window_ith+1.
    """

    for start_row in range(0, batch_size, effective_batch_size):
        # Lấy mini-batch nhỏ
        batch_rows = rep_shuffled[start_row:start_row+effective_batch_size]

        # Tính số chunk mỗi hàng
        num_chunks = (batch_rows.size(1) - 1) // timesteps

        for chunk_idx in range(num_chunks):
            # Lấy chunk input và target
            i = chunk_idx * timesteps
            inputs = batch_rows[:, i:i+timesteps]
            targets = batch_rows[:, i+1:i+1+timesteps]

            outputs, states = model(inputs, states)
            loss = loss_fn(outputs, targets.reshape(-1))
            loss = loss / accum_steps  # scale loss cho gradient accumulation
            loss.backward()

            # Reset hidden state nếu cần (truncated BPTT)
            states = (states[0].detach(), states[1].detach())

        # Sau accum_steps batch nhỏ, update optimizer
        clip_grad_norm(model.parameters(), 0.5)
        optimizer.step()
        optimizer.zero_grad()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

/tmp/ipython-input-2426329615.py:45: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  clip_grad_norm(model.parameters(), 0.5)


Epoch [1/20], Loss: 1.9380
Epoch [2/20], Loss: 1.7975
Epoch [3/20], Loss: 1.5905
Epoch [4/20], Loss: 1.5978
Epoch [5/20], Loss: 1.5815
Epoch [6/20], Loss: 1.5466
Epoch [7/20], Loss: 1.5449
Epoch [8/20], Loss: 1.5012
Epoch [9/20], Loss: 1.4434
Epoch [10/20], Loss: 1.3657
Epoch [11/20], Loss: 1.3876
Epoch [12/20], Loss: 1.3016
Epoch [13/20], Loss: 1.3104
Epoch [14/20], Loss: 1.2780
Epoch [15/20], Loss: 1.2441
Epoch [16/20], Loss: 1.2056
Epoch [17/20], Loss: 1.1163
Epoch [18/20], Loss: 1.0495
Epoch [19/20], Loss: 1.0149
Epoch [20/20], Loss: 1.0063


# IV. Test model

In [ ]:
# Test the model
with torch.no_grad():
    with open('results.txt', 'w') as f:
        # Set intial hidden ane cell states
        state = (torch.zeros(num_layers, 1, hidden_size),
                 torch.zeros(num_layers, 1, hidden_size))
        # Select one word id randomly and convert it to shape (1,1)
        input = torch.randint(0,vocab_size, (1,)).long().unsqueeze(1)

        for i in range(500):
            output, _ = model(input, state)
            print(output.shape)
            # Sample a word id from the exponential of the output
            prob = output.exp()
            word_id = torch.multinomial(prob, num_samples=1).item()
            print(word_id)
            # Replace the input with sampled word id for the next time step
            input.fill_(word_id)

            # Write the results to file
            word = corpus.dictionary.idx2word[word_id]
            word = '\n' if word == '<eos>' else word + ' '
            f.write(word)

            if (i+1) % 100 == 0:
                print('Sampled [{}/{}] words and save to {}'.format(i+1, 500, 'results.txt'))